# DVAE PPD Notebook

PyTorch DVAE baseline aligned with ECTRM data format.
Exports `.mat` with `beta`, `train_theta`, `test_theta`.


## 1) Imports


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import scipy
import scipy.io
import scipy.sparse as sp
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn import metrics

print('Python:', __import__('sys').version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## 2) Dataset discovery


In [ ]:
TARGET_DATASETS = ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
K_LIST = [20, 50, 100]
RANDOM_SEED = 42

REQUIRED_DATA_FILES = [
    'train_bow.npz', 'test_bow.npz', 'train_labels.txt', 'test_labels.txt', 'vocab.txt', 'word_embeddings.npz'
]

IS_KAGGLE = Path('/kaggle').exists()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ECTRM_source' / 'ECRTM' / 'data').exists():
            return candidate
    return start


if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working')
    INPUT_ROOT = Path('/kaggle/input')
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    INPUT_ROOT = PROJECT_ROOT


def has_required_files(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / f).exists() for f in REQUIRED_DATA_FILES)


def guess_dataset_name(path: Path):
    s = str(path).lower()
    if '20ng' in s or '20news' in s:
        return '20NG'
    if 'agnews' in s or 'ag_news' in s:
        return 'AGNews'
    if 'yahoo' in s:
        return 'YahooAnswer'
    if 'imdb' in s:
        return 'IMDB'
    return None


def iter_candidate_dirs(base: Path, max_depth: int = 5):
    if not base.exists():
        return
    base_depth = len(base.parts)
    for root, dirs, _ in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.parts) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        yield root_path


def discover_dataset_dirs():
    found = {}

    local_data_root = PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'data'
    for ds in TARGET_DATASETS:
        p = local_data_root / ds
        if has_required_files(p):
            found[ds] = p

    scan_roots = [Path('/kaggle/input'), PROJECT_ROOT, INPUT_ROOT] if IS_KAGGLE else [PROJECT_ROOT, INPUT_ROOT]
    seen = set()
    for scan_root in scan_roots:
        for cand in iter_candidate_dirs(scan_root):
            cs = str(cand)
            if cs in seen:
                continue
            seen.add(cs)
            if not has_required_files(cand):
                continue
            ds = guess_dataset_name(cand)
            if ds is not None and ds not in found:
                found[ds] = cand
    return found


DATASET_DIRS = discover_dataset_dirs()
OUTPUT_DIR = (Path('/kaggle/working') / 'Sortie_mat' / 'output' / 'kaggle' / 'DVAE') if IS_KAGGLE else (PROJECT_ROOT / 'Sortie_mat' / 'DVAE')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Resolved datasets:')
for ds in TARGET_DATASETS:
    print('-', ds, DATASET_DIRS.get(ds))


## 3) Utils and model


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

DVAE_DEFAULTS = {
    'epochs': 120,
    'batch_size': 200,
    'lr': 1e-3,
    'hidden_size': 256,
    'kl_weight': 0.5,
    'prior_alpha': 1.0,
}


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_vocab(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]


def load_bow_csr(path: Path):
    return sp.load_npz(path).astype(np.float32).tocsr()


class DVAETopicModel(nn.Module):
    def __init__(self, vocab_size, num_topics, hidden_size=256, prior_alpha=1.0):
        super().__init__()
        self.num_topics = num_topics
        self.prior_alpha = prior_alpha

        self.encoder = nn.Sequential(
            nn.Linear(vocab_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_topics),
        )
        self.beta_logits = nn.Parameter(torch.randn(num_topics, vocab_size) * 0.01)

    def encode_alpha(self, x):
        return torch.nn.functional.softplus(self.encoder(x)) + 1e-4

    def get_beta(self):
        return torch.softmax(self.beta_logits, dim=1)

    def sample_theta(self, alpha):
        if self.training:
            return torch.distributions.Dirichlet(alpha).rsample()
        return alpha / alpha.sum(dim=1, keepdim=True)

    def forward(self, x_raw, kl_weight=0.5):
        x_norm = x_raw / torch.clamp(x_raw.sum(dim=1, keepdim=True), min=1e-12)
        alpha = self.encode_alpha(x_norm)
        theta = self.sample_theta(alpha)

        beta = self.get_beta()
        recon = torch.matmul(theta, beta)
        recon_loss = -(x_raw * torch.log(recon + 1e-10)).sum(dim=1).mean()

        prior = torch.full_like(alpha, float(self.prior_alpha))
        kl = torch.distributions.kl_divergence(
            torch.distributions.Dirichlet(alpha),
            torch.distributions.Dirichlet(prior),
        ).mean()

        return recon_loss + kl_weight * kl

    def infer_theta(self, x_raw):
        self.eval()
        with torch.no_grad():
            x_norm = x_raw / torch.clamp(x_raw.sum(dim=1, keepdim=True), min=1e-12)
            alpha = self.encode_alpha(x_norm)
            theta = alpha / alpha.sum(dim=1, keepdim=True)
        return theta


## 4) Train and export


In [ ]:
def train_one_dvae(dataset, K, seed=42, cfg=None):
    cfg = dict(DVAE_DEFAULTS if cfg is None else cfg)

    if dataset not in DATASET_DIRS:
        raise KeyError(f'Dataset {dataset} not found')

    set_seed(seed)
    data_dir = DATASET_DIRS[dataset]

    train_csr = load_bow_csr(data_dir / 'train_bow.npz')
    test_csr = load_bow_csr(data_dir / 'test_bow.npz')

    x_train = torch.from_numpy(train_csr.toarray().astype(np.float32))
    x_test = torch.from_numpy(test_csr.toarray().astype(np.float32))

    model = DVAETopicModel(
        vocab_size=x_train.shape[1],
        num_topics=K,
        hidden_size=int(cfg['hidden_size']),
        prior_alpha=float(cfg['prior_alpha']),
    ).to(DEVICE)

    loader = DataLoader(TensorDataset(x_train), batch_size=int(cfg['batch_size']), shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=float(cfg['lr']))

    print(f'\n=== DVAE {dataset} K={K} ===')
    for epoch in range(int(cfg['epochs'])):
        model.train()
        losses = []
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            loss = model(batch, kl_weight=float(cfg['kl_weight']))
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:03d}/{cfg["epochs"]} - loss={np.mean(losses):.4f}')

    with torch.no_grad():
        beta = model.get_beta().cpu().numpy().astype(np.float32)
        train_theta = model.infer_theta(x_train.to(DEVICE)).cpu().numpy().astype(np.float32)
        test_theta = model.infer_theta(x_test.to(DEVICE)).cpu().numpy().astype(np.float32)

    ds_out = OUTPUT_DIR / dataset
    ds_out.mkdir(parents=True, exist_ok=True)
    out_path = ds_out / f'{dataset}_DVAE_K{K}_seed{seed}.mat'

    scipy.io.savemat(str(out_path), {'beta': beta, 'train_theta': train_theta, 'test_theta': test_theta, 'traintheta': train_theta, 'testtheta': test_theta})
    print('Saved:', out_path)
    return out_path


## 5) Run and metrics


In [ ]:
import pandas as pd


def topic_diversity_from_beta(beta, topn=25):
    all_words = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        all_words.extend(top_idx.tolist())
    return len(set(all_words)) / max(1, len(all_words))


def purity_score(y_true, y_pred):
    contingency_matrix = metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)


for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        print('SKIP missing dataset:', dataset)
        continue
    for K in K_LIST:
        train_one_dvae(dataset, K=K, seed=RANDOM_SEED)

rows = []
TOPN = 15
for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        continue

    labels = np.loadtxt(DATASET_DIRS[dataset] / 'test_labels.txt', dtype=np.int32)
    vocab = load_vocab(DATASET_DIRS[dataset] / 'vocab.txt')
    ds_out = OUTPUT_DIR / dataset

    for K in K_LIST:
        mat_path = ds_out / f'{dataset}_DVAE_K{K}_seed{RANDOM_SEED}.mat'
        if not mat_path.exists():
            continue

        loaded = scipy.io.loadmat(str(mat_path))
        beta = loaded['beta']
        test_theta = loaded['test_theta']

        n_eval = min(len(labels), test_theta.shape[0])
        y_true = labels[:n_eval]
        y_pred = np.argmax(test_theta[:n_eval], axis=1)

        rows.append({
            'model': 'DVAE',
            'dataset': dataset,
            'K': K,
            'Purity': float(purity_score(y_true, y_pred)),
            'NMI': float(metrics.cluster.normalized_mutual_info_score(y_true, y_pred)),
            'TD_top25': float(topic_diversity_from_beta(beta, topn=25)),
        })

        txt_path = ds_out / f'topics_for_cv_{dataset}_DVAE_K{K}_seed{RANDOM_SEED}.txt'
        with open(txt_path, 'w', encoding='utf-8') as f:
            for k in range(beta.shape[0]):
                top_idx = np.argsort(beta[k])[::-1][:TOPN]
                f.write(' '.join(vocab[i] for i in top_idx) + '\n')
        print('Saved:', txt_path)

if rows:
    df = pd.DataFrame(rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df)
    csv_path = OUTPUT_DIR / 'DVAE_metrics_TD_Purity_NMI.csv'
    df.to_csv(csv_path, index=False)
    print('Saved:', csv_path)
else:
    print('No DVAE .mat results found yet')
